# Air Quality Instruction-Tuning Dataset Builder

Takes the raw **AirQualityUCI CSV** and produces a diversified **instruction/output**
dataset (JSON + JSONL) suitable for fine-tuning a small language model, plus a
compressed copy of the JSONL for distribution/storage.

## Why this replaces the original `Data_Convertion.ipynb`

That notebook had three problems that would actively hurt a model trained on it:

1. **Only one instruction template**, repeated for every row verbatim
   (`"Predict the air quality metrics on {Date} at {Time}."`) — zero phrasing
   diversity means the model overfits to that exact sentence structure instead
   of learning to answer air-quality questions generally.
2. **Only the first 100 rows** of a 9,357-row dataset were used
   (`df.head(100)`) — far too little data and no temporal/seasonal variety.
3. **The dataset's missing-value sentinel (`-200`) was passed straight through
   as if it were a real reading** (e.g. `"CO(GT): -200.0 mg/m^3"`). A model
   trained on that learns `-200` is a plausible air-quality value, which is
   exactly the kind of **false-positive / fabricated-value** behavior we want
   to avoid.

## What this notebook does differently

- Replaces `-200` (and near-neighbors) with real missing values during
  cleaning, instead of leaving them as literal numbers.
- Grounds prompt phrasing and pollutant terminology in the actual **Air
  Quality API documentation** (pollutant names/units, supported health
  populations, the fact that raw sensor data carries no official AQI) rather
  than inventing wording from scratch.
- Generates **many diverse instruction templates** per row across several
  *intents* — point readings, multi-pollutant summaries, environmental
  (temp/humidity) queries, day-level aggregates, hour-over-hour trends, and
  health-context questions — each with multiple paraphrased surface forms.
- Explicitly generates **abstention/negative examples**: questions about a
  reading that's missing for that timestamp, and questions about pollutants
  the dataset never measured at all (PM2.5, SO2, NH3, ...). Training on
  correct "I don't have that" answers is the most direct way to reduce
  false-positive (fabricated) responses at inference time.
- Keeps the `PT08.*` tin-oxide **sensor response** columns clearly labeled as
  raw sensor signals, distinct from the `*(GT)` **reference-analyzer
  concentrations** — conflating the two would itself be a false claim.
- Uses the **full dataset** (configurable), not a fixed 100-row slice.
- Compresses the final JSONL to `.zst` (falls back to `.gz` if the
  `zstandard` package isn't available in your environment).

## How to use this notebook

1. Set `INPUT_CSV` below to your `AirQualityUCI.csv` path (works with Colab's
   `/content/...` or a local path).
2. Run all cells top to bottom.
3. Outputs land next to the input: a cleaned `AirQualityUCI_clean.json`, a
   training `air_quality_training.jsonl`, a small `*.val.jsonl` held-out
   split, and a compressed `air_quality_training.jsonl.zst` (or `.gz`).


## 1. Setup


In [ ]:
# zstandard gives much better compression than gzip at similar speed; it's optional —
# the compression cell below automatically falls back to gzip if this isn't available
# or if you're offline (e.g. this sandbox has no network access).
try:
    import zstandard  # noqa: F401
except ImportError:
    try:
        %pip install -q zstandard
    except Exception as e:
        print(f"Could not install zstandard ({e}); the compression cell will fall back to gzip.")


In [ ]:
import json
import random
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


## 2. Configuration

Adjust these to point at your data and control dataset size/composition.


In [ ]:
# --- Paths -----------------------------------------------------------------
INPUT_CSV = "/content/AirQualityUCI.csv"     # change this to your CSV's location
OUTPUT_DIR = Path("/content")                # where outputs are written

OUTPUT_JSON = OUTPUT_DIR / "AirQualityUCI_clean.json"
OUTPUT_JSONL = OUTPUT_DIR / "air_quality_training.jsonl"   # train split
# (the matching validation split is written automatically as
#  air_quality_training.val.jsonl next to it)

# --- Generation knobs --------------------------------------------------------
EXAMPLES_PER_ROW = 3     # how many diverse instruction/output pairs to sample per data row
VAL_FRACTION = 0.05      # held-out validation fraction
SEED = 42                # reproducibility

MISSING_SENTINEL = -200  # this dataset's documented missing-value code


## 3. Pollutant metadata & prompt diversity banks

Grounded in the Air Quality API docs (`03-pollutants.md`, `08-health-recommendations.md`,
`06-faq.md`): correct display names/units per column, which columns are calibrated
reference-analyzer **concentrations** vs. raw tin-oxide **sensor responses**, which
real-world pollutants this dataset does *not* measure, and the population categories
used for health-context questions.


In [ ]:
# --- Pollutant / column metadata -------------------------------------------------
# Grounded in the Air Quality API docs (pollutants.md, health-recommendations.md):
# reference-analyzer concentrations vs. raw tin-oxide "PT08" sensor responses are
# NOT the same kind of quantity, and the dataset carries no calibrated AQI — the
# generator must never blur these distinctions, since doing so is exactly the
# kind of overconfident/false-positive claim we're trying to avoid.
POLLUTANT_META = {
    "CO(GT)":   {"display": "CO",   "full": "carbon monoxide",          "unit": "mg/m^3", "kind": "reference"},
    "NMHC(GT)": {"display": "NMHC", "full": "non-methane hydrocarbons", "unit": "µg/m^3", "kind": "reference"},
    "C6H6(GT)": {"display": "C6H6 (benzene)", "full": "benzene",        "unit": "µg/m^3", "kind": "reference"},
    "NOx(GT)":  {"display": "NOx",  "full": "nitrogen oxides",          "unit": "ppb",    "kind": "reference"},
    "NO2(GT)":  {"display": "NO2",  "full": "nitrogen dioxide",         "unit": "µg/m^3", "kind": "reference"},
    "PT08.S1(CO)":   {"display": "PT08.S1 sensor (CO-targeted)",   "full": "tin-oxide sensor response targeting CO",        "unit": "sensor units", "kind": "sensor", "targets": "CO"},
    "PT08.S2(NMHC)": {"display": "PT08.S2 sensor (NMHC-targeted)", "full": "titania sensor response targeting NMHC",        "unit": "sensor units", "kind": "sensor", "targets": "NMHC"},
    "PT08.S3(NOx)":  {"display": "PT08.S3 sensor (NOx-targeted)",  "full": "tungsten-oxide sensor response targeting NOx",  "unit": "sensor units", "kind": "sensor", "targets": "NOx"},
    "PT08.S4(NO2)":  {"display": "PT08.S4 sensor (NO2-targeted)",  "full": "tungsten-oxide sensor response targeting NO2",  "unit": "sensor units", "kind": "sensor", "targets": "NO2"},
    "PT08.S5(O3)":   {"display": "PT08.S5 sensor (O3-targeted)",   "full": "indium-oxide sensor response targeting O3",     "unit": "sensor units", "kind": "sensor", "targets": "O3"},
    "T":  {"display": "temperature",        "full": "ambient temperature",   "unit": "°C", "kind": "env"},
    "RH": {"display": "relative humidity",  "full": "relative humidity",     "unit": "%",  "kind": "env"},
    "AH": {"display": "absolute humidity",  "full": "absolute humidity",     "unit": "g/m^3", "kind": "env"},
}
REFERENCE_COLS = [c for c, m in POLLUTANT_META.items() if m["kind"] == "reference"]
SENSOR_COLS = [c for c, m in POLLUTANT_META.items() if m["kind"] == "sensor"]
ENV_COLS = [c for c, m in POLLUTANT_META.items() if m["kind"] == "env"]

# Pollutants the Air Quality API documents (pollutants.md) that this dataset does
# NOT measure at all. Asking about these and training the model to correctly say
# "I don't have that" is one of the most direct anti-false-positive techniques:
# it stops the model from fabricating a plausible-looking number for something
# it was never given data on.
UNSUPPORTED_POLLUTANTS = ["PM2.5", "PM10", "SO2", "directly-measured ozone (O3 concentration)",
                           "ammonia (NH3)", "total reduced sulfur (TRS)", "photochemical oxidants (OX)"]


def indefinite_article(phrase):
    """"a"/"an" for a pollutant phrase, correctly handling acronyms that are
    read out letter-by-letter (e.g. "SO2" -> "ess-oh-two" -> "an SO2 reading",
    not "a SO2 reading") as distinct from ordinary words."""
    import re as _re
    first_token = phrase.split()[0] if phrase.split() else phrase
    is_acronym = bool(_re.fullmatch(r"[A-Z0-9.]+", first_token))
    # Letter *names* that start with a vowel sound: A E F H I L M N O R S X
    vowel_sound_letters = set("AEFHILMNORSX")
    first_char = first_token[:1].upper()
    if is_acronym:
        return "an" if first_char in vowel_sound_letters else "a"
    return "an" if first_char.lower() in "aeiou" else "a"

# From health-recommendations.md's "Supported populations" table.
HEALTH_POPULATIONS = [
    "the general population", "elderly people", "people with lung disease or asthma",
    "people with heart disease", "athletes doing strenuous outdoor exercise",
    "pregnant women", "young children",
]


## 4. Load & clean the CSV

- Handles this dataset's usual `;`-separated / `,`-decimal export format, with a
  plain-CSV fallback.
- Drops fully-empty rows/trailing `Unnamed` columns (a known artifact of this
  dataset's CSV export).
- **Replaces the `-200` missing-value sentinel with real `NaN`s** across all
  numeric columns — this is the key fix that stops those placeholder values
  from ever being presented as if they were real readings.
- Parses `Date` (`DD/MM/YYYY`) + `Time` (`HH.MM.SS`) into a real datetime
  column, while keeping the originals for display.


In [ ]:
def load_and_clean(csv_path):
    try:
        df = pd.read_csv(csv_path, sep=";", decimal=",")
    except Exception:
        df = pd.read_csv(csv_path)

    df = df.dropna(how="all")
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

    numeric_cols = [c for c in POLLUTANT_META if c in df.columns]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c] <= (MISSING_SENTINEL + 1), c] = np.nan  # -200 sentinel -> NaN

    def parse_dt(row):
        try:
            date_str = f"{row['Date']} {str(row['Time']).replace('.', ':')}"
            return datetime.strptime(date_str, "%d/%m/%Y %H:%M:%S")
        except Exception:
            return pd.NaT

    df["_datetime"] = df.apply(parse_dt, axis=1)
    df = df.dropna(subset=["_datetime"]).sort_values("_datetime").reset_index(drop=True)
    return df


df = load_and_clean(INPUT_CSV)
print(f"Loaded and cleaned {len(df)} rows, {df.shape[1]} columns.")
for c in REFERENCE_COLS + SENSOR_COLS + ENV_COLS:
    if c in df.columns:
        n_missing = df[c].isna().sum()
        pct = 100 * n_missing / len(df)
        print(f"  {c:16s} missing: {n_missing:5d} / {len(df)} ({pct:.1f}%)")
df.head()


## 5. Prompt paraphrase banks

Multiple surface phrasings per intent, randomly sampled — this is the direct fix
for the original notebook's single repeated template.


In [ ]:
SINGLE_POINT_TEMPLATES = [
    "What was the {display} level on {date} at {time}?",
    "Can you tell me the {display} reading for {date} at {time}?",
    "I need the {display} value recorded at {time} on {date}.",
    "How much {full} was measured on {date} around {time}?",
    "Predict the air quality metrics for {date} at {time}: what was {display}?",
    "Give me the {display} measurement at {time} on {date}.",
    "At {time} on {date}, what did the {display} sensor show?",
]

MULTI_POINT_TEMPLATES = [
    "What were the {list} readings on {date} at {time}?",
    "Give me a summary of {list} for {date} at {time}.",
    "Predict the air quality metrics on {date} at {time}, including {list}.",
    "How did {list} look at {time} on {date}?",
]

MISSING_TEMPLATES = [
    "What was the {display} level on {date} at {time}?",
    "Do you have a {display} reading for {time} on {date}?",
    "What did the {display} sensor record at {time} on {date}?",
]

UNSUPPORTED_TEMPLATES = [
    "What was the {pollutant} level on {date} at {time}?",
    "Can you give me the {pollutant} reading for {date} at {time}?",
    "How much {pollutant} was in the air on {date} at {time}?",
]

ENV_TEMPLATES = [
    "What were the temperature and humidity on {date} at {time}?",
    "How hot and humid was it at {time} on {date}?",
]

DAY_AGG_TEMPLATES = [
    "What was the average {display} on {date}?",
    "What were the minimum and maximum {display} readings on {date}?",
    "Summarize the {display} levels for {date}.",
]

TREND_TEMPLATES = [
    "How did the {display} level change between {time_prev} and {time_now} on {date}?",
    "Did {display} go up or down from {time_prev} to {time_now} on {date}?",
]

HEALTH_TEMPLATES = [
    "Based on the readings at {time} on {date}, is it safe for {population} to be outside?",
    "Given the air quality on {date} at {time}, should {population} avoid outdoor activity?",
    "Is the air at {time} on {date} okay for {population}?",
]


def fmt_date_time(dt):
    return dt.strftime("%d/%m/%Y"), dt.strftime("%H:%M")


def fmt_value(col, value):
    meta = POLLUTANT_META[col]
    return f"{meta['display']}: {value:g} {meta['unit']}"


## 6. Example generators

Each function implements one *intent*. `gen_missing_abstention` and
`gen_unsupported_abstention` are the negative/refusal examples that directly target
false-positive responses; `gen_health_context` demonstrates reporting raw readings
**without** overclaiming an official safety verdict (this dataset has no calibrated
AQI, so a confident "yes it's safe" would itself be a false positive).


In [ ]:
def gen_single_point(row, rng):
    present = [c for c in REFERENCE_COLS + SENSOR_COLS + ENV_COLS if pd.notna(row.get(c))]
    if not present:
        return None
    col = rng.choice(present)
    meta = POLLUTANT_META[col]
    date, time = fmt_date_time(row["_datetime"])
    instr = rng.choice(SINGLE_POINT_TEMPLATES).format(display=meta["display"], full=meta["full"], date=date, time=time)
    value = row[col]
    caveat = " (this is a raw tin-oxide sensor response, not a calibrated concentration)" if meta["kind"] == "sensor" else ""
    output = f"{fmt_value(col, value)}{caveat}."
    return instr, output


def gen_multi_point(row, rng, k=None):
    present = [c for c in REFERENCE_COLS + ENV_COLS if pd.notna(row.get(c))]
    if len(present) < 2:
        return None
    k = k or rng.choice([2, 3])
    cols = rng.sample(present, min(k, len(present)))
    date, time = fmt_date_time(row["_datetime"])
    names = [POLLUTANT_META[c]["display"] for c in cols]
    list_str = ", ".join(names[:-1]) + (" and " + names[-1] if len(names) > 1 else names[0])
    instr = rng.choice(MULTI_POINT_TEMPLATES).format(list=list_str, date=date, time=time)
    output = "; ".join(fmt_value(c, row[c]) for c in cols) + "."
    return instr, output


def gen_missing_abstention(row, rng):
    all_cols = REFERENCE_COLS + SENSOR_COLS + ENV_COLS
    missing = [c for c in all_cols if c in row.index and pd.isna(row.get(c))]
    if not missing:
        return None
    col = rng.choice(missing)
    meta = POLLUTANT_META[col]
    date, time = fmt_date_time(row["_datetime"])
    instr = rng.choice(MISSING_TEMPLATES).format(display=meta["display"], date=date, time=time)
    output = (f"No valid {meta['display']} reading was recorded at {time} on {date} — "
              f"the sensor logged a missing-value code for this slot, so I can't give you a number for it.")
    return instr, output


def gen_unsupported_abstention(row, rng):
    pollutant = rng.choice(UNSUPPORTED_POLLUTANTS)
    date, time = fmt_date_time(row["_datetime"])
    instr = rng.choice(UNSUPPORTED_TEMPLATES).format(pollutant=pollutant, date=date, time=time)
    art = indefinite_article(pollutant)
    output = (f"This dataset doesn't include {pollutant} measurements. It only covers CO, NMHC, benzene (C6H6), "
              f"NOx, NO2, an ozone-correlated sensor response, plus temperature and humidity — "
              f"I can't provide {art} {pollutant} reading from it.")
    return instr, output


def gen_env(row, rng):
    if pd.isna(row.get("T")) or pd.isna(row.get("RH")):
        return None
    date, time = fmt_date_time(row["_datetime"])
    instr = rng.choice(ENV_TEMPLATES).format(date=date, time=time)
    output = f"{fmt_value('T', row['T'])}; {fmt_value('RH', row['RH'])}."
    return instr, output


def gen_health_context(row, rng):
    present = [c for c in REFERENCE_COLS if pd.notna(row.get(c))]
    if not present:
        return None
    cols = rng.sample(present, min(2, len(present)))
    population = rng.choice(HEALTH_POPULATIONS)
    date, time = fmt_date_time(row["_datetime"])
    instr = rng.choice(HEALTH_TEMPLATES).format(population=population, date=date, time=time)
    readings = "; ".join(fmt_value(c, row[c]) for c in cols)
    output = (f"At {time} on {date}, the recorded readings were: {readings}. "
              f"This dataset doesn't include an official air quality index or health-recommendation calculation, "
              f"so I can't give a certified safety verdict for {population} — please check a live air-quality/health "
              f"advisory service for that. I can only confirm what the sensors measured.")
    return instr, output


def gen_day_aggregate(day_df, rng):
    present = [c for c in REFERENCE_COLS + ENV_COLS if day_df[c].notna().any()]
    if not present:
        return None
    col = rng.choice(present)
    meta = POLLUTANT_META[col]
    series = day_df[col].dropna()
    date = day_df["_datetime"].iloc[0].strftime("%d/%m/%Y")
    instr = rng.choice(DAY_AGG_TEMPLATES).format(display=meta["display"], date=date)
    if series.empty:
        output = f"No valid {meta['display']} readings were available on {date}."
    else:
        output = (f"On {date}, {meta['display']} averaged {series.mean():.2f} {meta['unit']} "
                  f"(min {series.min():g}, max {series.max():g}, n={len(series)} readings).")
    return instr, output


def gen_trend(df_sorted, i, rng):
    if i == 0:
        return None
    row_now, row_prev = df_sorted.iloc[i], df_sorted.iloc[i - 1]
    present = [c for c in REFERENCE_COLS + ENV_COLS if pd.notna(row_now.get(c)) and pd.notna(row_prev.get(c))]
    if not present:
        return None
    col = rng.choice(present)
    meta = POLLUTANT_META[col]
    date, time_now = fmt_date_time(row_now["_datetime"])
    _, time_prev = fmt_date_time(row_prev["_datetime"])
    instr = rng.choice(TREND_TEMPLATES).format(display=meta["display"], time_prev=time_prev, time_now=time_now, date=date)
    v_prev, v_now = row_prev[col], row_now[col]
    delta = v_now - v_prev
    direction = "increased" if delta > 0 else ("decreased" if delta < 0 else "stayed the same")
    output = (f"{meta['display']} {direction} from {v_prev:g} to {v_now:g} {meta['unit']} "
              f"between {time_prev} and {time_now} on {date}.")
    return instr, output


## 7. Generate the dataset

Weighted sampling across intents, with abstention examples (missing-value +
unsupported-pollutant) deliberately included at meaningful volume so the model
sees negative/refusal cases, not just positive factual ones.


In [ ]:
def generate_dataset(df, examples_per_row=EXAMPLES_PER_ROW, seed=SEED):
    rng = random.Random(seed)
    examples = []

    row_generators = (
        [gen_single_point] * 3
        + [gen_multi_point] * 2
        + [gen_env] * 1
        + [gen_missing_abstention] * 2
        + [gen_unsupported_abstention] * 2
        + [gen_health_context] * 1
    )

    for _, row in df.iterrows():
        chosen = rng.sample(row_generators, k=min(examples_per_row, len(row_generators)))
        for fn in chosen:
            result = fn(row, rng)
            if result:
                examples.append({"instruction": result[0], "output": result[1], "_source": fn.__name__})

    # Day-level aggregates: one per calendar day
    for _, day_df in df.groupby(df["_datetime"].dt.date):
        result = gen_day_aggregate(day_df, rng)
        if result:
            examples.append({"instruction": result[0], "output": result[1], "_source": "gen_day_aggregate"})

    # Hour-over-hour trend comparisons: sampled, not exhaustive
    trend_sample_idx = rng.sample(range(1, len(df)), k=min(len(df) // 4, 2000))
    for i in trend_sample_idx:
        result = gen_trend(df, i, rng)
        if result:
            examples.append({"instruction": result[0], "output": result[1], "_source": "gen_trend"})

    rng.shuffle(examples)
    return examples


examples = generate_dataset(df, examples_per_row=EXAMPLES_PER_ROW, seed=SEED)
ratio = len(examples) / len(df)
print(f"Generated {len(examples)} instruction/output examples from {len(df)} source rows ({ratio:.1f}x).")
print(Counter(e["_source"] for e in examples))


## 8. Write outputs: cleaned JSON + train/val JSONL


In [ ]:
def write_outputs(df, examples, json_path, jsonl_path, val_fraction=VAL_FRACTION, seed=SEED):
    # Full cleaned dataset as JSON (nulls instead of -200 sentinels; ISO datetime added)
    clean_df = df.drop(columns=["_datetime"]).assign(
        datetime_iso=df["_datetime"].dt.strftime("%Y-%m-%dT%H:%M:%S")
    )
    clean_json = json.loads(clean_df.to_json(orient="records", date_format="iso"))
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(clean_json, f, indent=2, ensure_ascii=False)

    rng = random.Random(seed)
    idx = list(range(len(examples)))
    rng.shuffle(idx)
    n_val = int(len(idx) * val_fraction)
    val_idx = set(idx[:n_val])

    jsonl_path = str(jsonl_path)
    train_path = jsonl_path
    val_path = jsonl_path.replace(".jsonl", ".val.jsonl")
    with open(train_path, "w", encoding="utf-8") as ftr, open(val_path, "w", encoding="utf-8") as fval:
        for i, ex in enumerate(examples):
            clean_ex = {"instruction": ex["instruction"], "output": ex["output"]}
            line = json.dumps(clean_ex, ensure_ascii=False) + "\n"
            (fval if i in val_idx else ftr).write(line)

    return train_path, val_path


train_path, val_path = write_outputs(df, examples, OUTPUT_JSON, OUTPUT_JSONL)
print(f"Wrote cleaned JSON:  {OUTPUT_JSON}")
print(f"Wrote train JSONL:   {train_path} ({sum(1 for _ in open(train_path))} lines)")
print(f"Wrote val JSONL:     {val_path} ({sum(1 for _ in open(val_path))} lines)")


## 9. Preview a sample


In [ ]:
with open(train_path, "r", encoding="utf-8") as f:
    preview = [json.loads(next(f)) for _ in range(10)]
pd.DataFrame(preview)


## 10. Compress the JSONL for distribution

Uses `zstandard` (`.jsonl.zst`) when available — better compression ratio and
speed than gzip, and the de-facto standard for LLM training corpora — and
automatically falls back to gzip (`.jsonl.gz`) if the package isn't installed
or installable (e.g. no network access) in the current environment.


In [ ]:
import gzip
import os
import shutil


def compress_jsonl(path):
    path = str(path)
    orig_size = os.path.getsize(path)
    try:
        import zstandard as zstd
        out_path = path + ".zst"
        cctx = zstd.ZstdCompressor(level=19)
        with open(path, "rb") as fin, open(out_path, "wb") as fout:
            cctx.copy_stream(fin, fout)
        method = "zstandard (level 19)"
    except ImportError:
        out_path = path + ".gz"
        with open(path, "rb") as fin, gzip.open(out_path, "wb", compresslevel=9) as fout:
            shutil.copyfileobj(fin, fout)
        method = "gzip (level 9, zstandard not available)"

    comp_size = os.path.getsize(out_path)
    ratio = 100 * comp_size / orig_size
    print(f"{path}")
    print(f"  -> {out_path}")
    print(f"  {orig_size:,} bytes -> {comp_size:,} bytes ({ratio:.1f}% of original, via {method})")
    return out_path


compressed_train_path = compress_jsonl(train_path)
compressed_val_path = compress_jsonl(val_path)


## Notes / next steps

- **Tune volume vs. diversity**: raise `EXAMPLES_PER_ROW` for more examples per
  timestamp, or thin out a generator in the `row_generators` weighting in
  section 7 if one intent feels over-represented.
- **Abstention ratio**: the `_source` counts printed in section 7 show how many
  examples are missing-value or unsupported-pollutant abstentions. For a small
  model, keeping this around 20–35% of the dataset (as configured) teaches
  correct refusal behavior without drowning out the factual examples that make
  the model actually useful.
- **Swap in your own CSV**: as long as it has the same column names as the UCI
  AirQualityUCI dataset, nothing else needs to change — just update
  `INPUT_CSV`.
- **Loading for training**: both the `.jsonl` and compressed `.jsonl.zst`/`.gz`
  can be streamed directly with Hugging Face `datasets`
  (`load_dataset("json", data_files=...)` handles `.gz` natively; for `.zst`,
  decompress first or use a streaming reader that supports it).
